# 低比特量化

> 上一章考察了解码阶段，明确了 Decode 的瓶颈在数据传输：权重在每一步推理中都必须被完整读取。
>
> 假设我们试图在 16 GB 显存的显卡上运行 14B 模型。BF16 权重需要 28 GB 空间，显然无法容纳；若将每个参数压缩至 4 bit，体积仅剩 7 GB，便可放入显存。这种「4 倍缩水」的计算十分直观，但关键疑问在于：数值被削减如此之多，模型为何仍能工作？精度下降到何种程度会失效？换句话说，比特数被砍去大半，模型依靠什么避免彻底瘫痪？
>
> 量化通过缩减数据传输量，同时改善显存占用与 TPOT。本章将解答这一核心疑问，并介绍量化的四组核心内容：
>
> 1. **INT4 的基本原理**：16 个离散档位如何容纳浮点数——scale 与粒度。
> 2. **整数与浮点格式**：除 INT4 / INT8 外，FP8 与 FP4 存在的理由。
> 3. **算法与记号**：GPTQ、AWQ、SmoothQuant 分别修复什么问题，`W4A16` / `W8A8` 的含义。
> 4. **生态与实战**：常见格式如何自行制作、依据部署目标如何选择，量化的收益与代价。
>
> 读完本章，你将理解 Hugging Face、vLLM、SGLang、llama.cpp 文档中的 `W4A16`、`FP8`、`Q4_K_M` 等标记所指为何。归根结底，它们改变的是「以更少比特表示模型」的方式，并未修改模型结构本身。

下面我们先从最直观的存储账目开始分析。

## 1. 模型大小与精度

推理过程中，显存消耗主要来自两个部分。其一是权重，其大小固定，且在 Decode 阶段每步均需搬运；其二是 KV Cache，它随上下文长度增加而扩张。

为何量化首先针对权重？因为权重既是显存占用的主体，又是前文所述每步必搬的数据。换句话说，权重同时具备高体积与高频搬运的特征，优先压缩它收益最大。

不同精度下权重占用空间如何？我们紧接着用具体数值对比说明。

In [ ]:
def weight_size_gb(params_billion, bits):
    """按参数量和每参数比特数估算权重显存（GB）"""
    return params_billion * 1e9 * bits / 8 / 1e9

for size in [7, 70]:
    for bits in [16, 8, 4]:
        print(f"{size:>2d}B @ {bits:2d}-bit -> {weight_size_gb(size, bits):6.1f} GB")

print()
print("关键观察：70B BF16 约 140 GB，单卡放不下；压到 4-bit 约 35 GB，一张卡就装下了")

In [ ]:
# 显存地图：参数量 × 精度决定每个 Decode step 要搬多少权重
import matplotlib.pyplot as plt

bits_options = [16, 8, 4]
sizes = [7, 70]
xs = range(len(bits_options))
width = 0.35

plt.figure(figsize=(6.5, 3.5))
for j, size in enumerate(sizes):
    values = [weight_size_gb(size, b) for b in bits_options]
    plt.bar([x + (j - 0.5) * width for x in xs], values, width=width,
            label=f"{size}B params")
plt.xticks(list(xs), [f"{b}-bit" for b in bits_options])
plt.ylabel("weight size (GB)")
plt.title("Smaller dtype = less memory to load every decode step")
plt.legend()
plt.show()

这张图展示了量化的价值。但一个更基础的疑问随之产生：16 bit 浮点具备超过 6 万种取值，而 4 bit 整数仅有 16 种。换句话说，可表示数值数量急剧收缩，模型为何尚未崩溃？

要解答此问题，需先明确「量化」的操作定义。我们从实现层面逐步拆解。

## 2. 从浮点到 INT4

INT4 指每个数值仅占用 4 个 bit，总共提供 16 个可选离散值（对称量化通常采用 -7 到 +7）。然而模型中的权重段包含数十亿浮点数，如何纳入这 16 个档位？

换句话说，量化并非简单将小数四舍五入为整数，而是将连续取值区间强制映射至 16 个固定等级。相邻等级间的步长称为 **scale**。

对称量化的公式长这样：

$$
q = \mathrm{clip}(\mathrm{round}(x/s),\ q_{min},\ q_{max}),
\qquad \hat{x} = s \cdot q
$$

我们通过两个手算实例体会此过程。从 7 个数中选取 `-0.72` 与 `0.63`，设定 `scale = 1.0 / 7 ≈ 0.143`：

```text
-0.72 / 0.143 ≈ -5.04  ->  round 成 -5  ->  反量化 -5 × 0.143 ≈ -0.714，误差约 0.006
 0.63 / 0.143 ≈  4.41  ->  round 成  4  ->  反量化  4 × 0.143 ≈  0.571，误差约 0.059
```

每个数值被牵引至最近档位，最大误差不超过半个格子（scale / 2）。孤立看单点误差微小，但整份权重包含数百万数值，误差累积便是模型性能下降的根源。后续量化算法均以降低累计误差为目标。

In [ ]:
import numpy as np

x = np.array([-1.0, -0.72, -0.31, 0.0, 0.18, 0.63, 1.0], dtype=np.float32)
qmax = 7
scale = np.max(np.abs(x)) / qmax
q = np.round(x / scale).clip(-qmax, qmax).astype(np.int32)
x_hat = q * scale

print("scale:", round(float(scale), 4))
print("float:", x)
print("INT4 :", q)
print("反量化:", np.round(x_hat, 3))
print()
print(f"关键观察：最大误差 {np.max(np.abs(x - x_hat)):.4f} < scale/2 = {scale/2:.4f}")
print("每个数都被拉到最近的格子——量化误差就是这几百万次拉扯的累加")

## 3. 量化粒度

若整个权重层共享单一 scale，相当于假定所有数值范围相近。实际情况如何？不同 channel 的数值范围往往相差数倍。

倘若某 outlier 通道数值极大，共享 scale 将被迫放大，其余通道只能勉强挤入少数粗糙档位。如何应对？自然的选择是细化 scale 的分配。

```text
per-tensor  : 整个张量一个 scale
per-channel : 每个 channel 一个 scale（outlier 只污染自己）
per-group   : 每 128 个数一个 scale（现代 4-bit 方案的默认粒度）
```

粒度越细，误差越低，但副作用是 scale 元数据增加、Kernel 实现更繁琐。我们使用一份含 outlier 的模拟权重，对三种粒度分别测试：

In [ ]:
np.random.seed(7)
W = np.random.randn(4, 16).astype(np.float32) * 0.25
W[1] *= 8  # 第 1 个 channel 是 outlier，范围被拉大 8 倍

def qdq_tensor(a):
    """整层共用一个 scale"""
    s = max(np.max(np.abs(a)) / 7, 1e-12)
    q = np.round(a / s).clip(-7, 7)
    return q * s

def qdq_channel(a):
    """每个 channel 一个 scale（按行）"""
    s = np.maximum(np.max(np.abs(a), axis=1, keepdims=True) / 7, 1e-12)
    q = np.round(a / s).clip(-7, 7)
    return q * s

def qdq_group(a, group=4):
    """每 4 个相邻元素一组，各用各的 scale"""
    out = np.zeros_like(a)
    for r in range(a.shape[0]):
        for c0 in range(0, a.shape[1], group):
            block = a[r, c0:c0 + group]
            s = max(np.max(np.abs(block)) / 7, 1e-12)
            out[r, c0:c0 + group] = np.round(block / s).clip(-7, 7) * s
    return out

errs = {}
for name, fn in [("per-tensor", qdq_tensor), ("per-channel", qdq_channel),
                 ("per-group(4)", lambda a: qdq_group(a, 4))]:
    errs[name] = float(np.mean(np.abs(W - fn(W))))
    print(f"{name:<14} MAE = {errs[name]:.4f}")

print()
print("关键观察：outlier 通道存在时，粒度越细，其他通道被连累得越轻")

In [ ]:
# 同一份权重、三种粒度的误差对比：越绿误差越小
import matplotlib.pyplot as plt

plt.figure(figsize=(5.5, 3.2))
plt.bar(errs.keys(), errs.values(), color=["tab:red", "tab:orange", "tab:green"])
plt.ylabel("MAE (lower is better)")
plt.title("Finer granularity -> smaller quantization error")
plt.show()

## 4. 权重与 Activation

此前我们压缩的对象均为权重。但模型前向计算时还存在另一要素：**Activation**——即每层的输入，它随 Prompt 内容动态变化。是否需将其一并压缩？这反映于记号中的第二个字母：

```text
W4A16 = Weight 4-bit, Activation 16-bit   <- 只压权重
W8A8  = Weight 8-bit, Activation 8-bit    <- 两者都压
W4A8  = Weight 4-bit, Activation 8-bit
```

为何主流方案优先采用 weight-only？因为两者难度悬殊。权重在推理时保持不变，可离线逐步量化、校准与补偿；activation 则随每个输入变动，不存在「离线」窗口，且常在固定少数通道出现大幅 outlier。

换句话说，权重状态静止故可离线处理，activation 动态多变难以捕捉，同一套离散格难以容纳所有输入。此差异恰是下一节三种算法的出发点：它们分别修复「量化误差」的不同维度。

## 5. GPTQ、AWQ 与 SmoothQuant

首先确立基准：直接取整至最近档位（RTN，Round-To-Nearest）即第 2 节的手算操作，亦是各类量化算法的 baseline。在 8-bit 下 RTN 已足够；但降至 4-bit 时，RTN 的累计误差便开始损害模型。三种算法均试图挽回此误差，但路径迥异。

GPTQ 的出发点为何？量化产生的误差能否补偿？它逐列量化权重，每完成一列便利用二阶信息（近似 Hessian）评估该列误差对输出的影响，将误差分摊至尚未量化的列以补偿。目标是使「量化后模型输出」逼近「量化前模型输出」。它属于 post-training、weight-only 方法。

AWQ 提出疑问：所有通道重要性相同吗？显然否。观察 activation 统计可见，少数通道数值极大，其上的量化误差被放大，对模型伤害最深。AWQ 依据 activation 分布为重要通道分配更精细 scale（等价于先放大再量化），无需反向传播，校准开销低于 GPTQ。

SmoothQuant 的思路：activation 难以量化，能否转移难度？activation 的 outlier 集中于固定通道，而权重易于量化。它执行等价变换：各通道 activation 除以平滑因子，对应权重乘以同因子。数学上输出不变，但数值难度从 activation 侧移至 weight 侧，双方终可量化。它是 W8A8 路线的代表。

把三个名字放回表格：

| 算法 | 救的问题 | 路线 |
|:---|:---|:---|
| GPTQ | 4-bit 权重的累计误差 | weight-only，二阶补偿 |
| AWQ | 重要通道被量化伤到 | weight-only，保护 salient channels |
| SmoothQuant | activation outlier 难量化 | W8A8，难度迁移 |

需注意它们是**量化算法**，而非格式。同一「4-bit」可由 RTN、GPTQ 或 AWQ 生成，质量可能存在明显差距。

## 6. PTQ、QAT 与 KV Cache 量化

存在两个常见分类维度，我们先厘清概念。

何时执行量化？PTQ（Post-Training Quantization）指训练完成后量化——本章内容均属此类：取现有 checkpoint、运行校准数据、生成低比特版本，无需训练资源，此为其普及原因。QAT（Quantization-Aware Training）则在训练或微调时插入「量化-反量化」模拟，使模型在损失函数中直接适应低比特误差，效果上限更高，但需投入完整训练成本。生产环境常见流程为：先试 PTQ，若评测掉分过多再启用 QAT。

KV Cache 量化针对何物？除权重外，前文「LLM 推理的计算与显存开销」已述：长上下文、多并发下 KV Cache 成为显存主要消耗。将 KV 从 FP16 压至 FP8 或 INT8，相当于将该部分开销再减半。它与 `W4A16` 压缩对象不同，引擎配置中常表示为 `kv_cache_dtype=fp8` 等参数。

至此前述格均为整数档位。另有一族量化选择浮点格式路径。

## 7. 浮点格式 FP8 与 FP4

INT4 / INT8 的档位是**均匀**的：相邻等级间距一致。但真实权重分布呈「大量数值聚集于 0 附近，少数 outlier 延伸至远处」。均匀格面对此形态便两头受损：0 附近数值挤作一团难以区分档位，远处稀疏区间却占据大片无用范围。

如何应对？浮点格式令档位**不均匀**：数值以「指数 + 尾数」表示，靠近 0 区域格密，远离 0 区域格稀。这恰好匹配「中间密、两头稀」的形态。

$$
\underbrace{\pm}_{符号}\ \underbrace{2^{e}}_{指数位}\ \underbrace{\times 1.m}_{尾数位}
$$

FP16、BF16、FP8、FP4 皆为该结构的不同位宽组合：

| 格式 | 比特 | 指数 / 尾数 | 出现场景 |
|:---|:---|:---|:---|
| FP16 | 16 | 5 / 10 | 传统混合精度训练 |
| BF16 | 16 | 8 / 7 | 现代训练默认，范围大 |
| FP8 E4M3 | 8 | 4 / 3 | 精度优先的推理 / 训练 |
| FP8 E5M2 | 8 | 5 / 2 | 范围优先，能装下更大的数值 |
| FP4 E2M1 | 4 | 2 / 1 | 最新一代硬件 |

E4M3 与 E5M2 的差异即此权衡：指数位多一位，可表示范围更大，但各区间档位变稀。

硬件是此类格式的真正驱动力：Hopper 架构 GPU（H100 / H200）原生支持 FP8 矩阵乘，Blackwell 架构进一步支持 FP4。DeepSeek-V3 自训练起便采用 FP8，开源模型直接发布 FP8 checkpoint 已成常态；FP4 生态（NVFP4、MXFP4——在 FP4 之上给每 16 / 32 个元素配一个 FP8 的 block scale）正随新硬件扩展。

浮点格相较整数格优势何在？无需死记结论，可通过实验验证：同一份正态分布权重，分别用 INT4（均匀 16 档）与 FP4 E2M1（不均匀 16 档）量化重构，比较误差。

In [ ]:
# FP4 E2M1 的 16 个档位：指数让 0 附近更密
# 正档位 {0, 0.5, 1, 1.5, 2, 3, 4, 6}，对称取负数
fp4_grid = np.array([-6, -4, -3, -2, -1.5, -1, -0.5,
                     0, 0.5, 1, 1.5, 2, 3, 4, 6])

def qdq_fp4(a):
    """量化到 E2M1 的浮点档位再还原"""
    s = np.max(np.abs(a)) / 6.0
    z = a / s
    idx = np.abs(z[..., None] - fp4_grid).argmin(axis=-1)
    return fp4_grid[idx] * s

def qdq_int4(a):
    """对照：对称 INT4，均匀 15 档（-7 到 +7，惯例对称不用 -8）"""
    s = np.max(np.abs(a)) / 7.0
    return np.round(a / s).clip(-7, 7) * s

np.random.seed(0)
W = np.random.randn(1000, 100).astype(np.float32) * 0.5

err_int4 = float(np.mean(np.abs(W - qdq_int4(W))))
err_fp4 = float(np.mean(np.abs(W - qdq_fp4(W))))
print(f"INT4 (uniform 15 levels)   MAE = {err_int4:.4f}")
print(f"FP4 E2M1 (float 16 levels) MAE = {err_fp4:.4f}")
print()
print("关键观察：同样 4 bit，浮点档位对「挤在 0 附近」的分布误差更小")
print("——这就是 FP8 / FP4 在新硬件上立足的底层原因")

In [ ]:
# 左：权重直方图（中间密两头稀）；右：两种格式的误差对比
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].hist(W.flatten(), bins=80, color="tab:gray", alpha=0.7, density=True)
axes[0].set_title("Weight distribution: dense near zero")
axes[0].set_xlabel("value")
axes[0].set_ylabel("density")

axes[1].bar(["INT4\n(uniform)", "FP4 E2M1\n(float)"], [err_int4, err_fp4],
            color=["tab:red", "tab:green"])
axes[1].set_ylabel("MAE (lower is better)")
axes[1].set_title("Same 4 bits, different grid shape")
plt.tight_layout()
plt.show()

## 8. GGUF 与 llama.cpp 生态

迄今，模型下载多为 Hugging Face 的目录形式：safetensors 分片、config、tokenizer 等多个文件。另一生态采用单文件路线：**GGUF**（GPT-Generated Unified Format）将权重、词表、配置全数打包入一文件，拷贝即用。它是 llama.cpp 生态的标准格式，Ollama、LM Studio 等本地工具底层均依赖它。

GGUF 模型名中的 `Q4_K_M` 蕴含实际信息，可按「比特 _ 结构 _ 档位」三段解析：

```text
Q4_K_M
 │  │  └─ M = Medium：部分关键层用更高精度（Q6_K），其余层用 Q4_K
 │  └──── K = K-quant：按 block 组织、带独立 scale 的量化结构
 └─────── 4 = 主权重的比特数
```

K-quant 家族做法与本章第 3 节 per-group 同源：每 32 个权重为一个 block，各带独立 scale，再每 256 个 block 组成 super-block 进一步压缩 scale 自身。家族按「体积-质量」排序：`Q2_K`（最小、损失最明显）、`Q3_K_S` / `Q3_K_M`、`Q4_K_S` / `Q4_K_M`、`Q5_K_M`、`Q6_K`、`Q8_0`（接近无损的基线）。经验上 `Q4_K_M` 是体积与质量的常用平衡点，亦属社区下载量最大档位之一。

进阶工具 `imatrix`（importance matrix）：利用校准语料统计各位置权重重要性，使量化误差向不重要位置集中。此思路与 AWQ 保护重要通道同源。llama.cpp 的 `llama-imatrix` 工具离线生成，量化时挂载即可。

何时选择 GGUF？无独立 GPU 的机器（纯 CPU / Apple Silicon）、显存极小、端侧设备，或仅需在笔记本快速运行模型。GPU 服务器高并发服务仍以 vLLM / SGLang 的 GPTQ / AWQ / FP8 生态为主流。二者非竞争关系，各自绑定自身运行时。

## 9. 量化方案的选择

将部署中实际面临的选择归纳于表格。逻辑分两步：先确认硬件类型（决定运行时），再考量精度预算（决定格式）：

| 部署目标 | 常见格式 | 运行时 / 工具链 |
|:---|:---|:---|
| NVIDIA GPU 高并发服务 | GPTQ / AWQ（INT4）、FP8 checkpoint | vLLM / SGLang / TensorRT-LLM |
| Blackwell 新硬件 | NVFP4 / MXFP4 | TensorRT-LLM / vLLM |
| 快速实验、显存紧张 | bitsandbytes nf4（`load_in_4bit`） | transformers |
| CPU / Mac / 端侧 | GGUF Q4_K_M 等 | llama.cpp / Ollama / LM Studio |
| 长上下文高并发 | W8A8 或 FP8 权重 + FP8 KV | vLLM / SGLang |

选型时决定成败的关键步骤是**查阅引擎支持矩阵**：同一「4-bit」，GPTQ 的 checkpoint 无法在 llama.cpp 运行，GGUF 亦不能进入 vLLM 主路径。为何如此严格？因为格式与运行时相互绑定。换句话说，选错引擎模型根本无法启动。应先定引擎，再选格式，最后考虑比特数。

## 10. 量化实战

前九节皆属原理。本节我们实操最常见的三条量化路径：生成 vLLM 可用的 GPTQ / FP8、AWQ，以及 llama.cpp 可用的 GGUF。这些命令为何均需 GPU？因为量化需执行校准数据。建议于独立终端运行。先概览三条路径全景：

| 路径 | 工具 | 产出 | 给谁跑 |
|:---|:---|:---|:---|
| GPTQ / FP8 | llm-compressor | HF 目录 checkpoint | vLLM / SGLang |
| AWQ | AutoAWQ | HF 目录 checkpoint | vLLM / SGLang |
| GGUF | llama.cpp 两件套 | 单文件 `.gguf` | llama.cpp / Ollama |

### 路径一 GPTQ 与 FP8（llm-compressor）

llm-compressor 是 vLLM 生态的官方量化工具。GPTQ 与 FP8 共用 `oneshot` 接口，仅替换 recipe。GPTQ 版本如下：

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor.transformers import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

model_id = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# W4A16 就是第 4 节的记号；lm_head 对精度最敏感，通常跳过不压
recipe = GPTQModifier(targets="Linear", scheme="W4A16", ignore=["lm_head"])
oneshot(model=model, tokenizer=tokenizer, recipe=recipe,
        output_dir="Qwen2.5-7B-Instruct-GPTQ-W4A16")
```

将 recipe 替换为下述内容即为 FP8：

```python
from llmcompressor.modifiers.quantization import QuantizationModifier

# FP8_DYNAMIC：权重离线量化，activation 用动态 scale（运行时统计）
recipe = QuantizationModifier(targets="Linear", scheme="FP8_DYNAMIC",
                              ignore=["lm_head"])
oneshot(model=model, tokenizer=tokenizer, recipe=recipe,
        output_dir="Qwen2.5-7B-Instruct-FP8")
```

`FP8_DYNAMIC` 即第 6 节「activation 随输入变化、难以离线量化」的工程解——activation 的 scale 不在离线阶段固定，而于运行时按张量统计。

### 路径二 AWQ（AutoAWQ）

```python
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

model_path = "Qwen/Qwen2.5-7B-Instruct"
model = AutoAWQForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4,
                "version": "GEMM"}
model.quantize(tokenizer, quant_config=quant_config)

model.save_quantized("Qwen2.5-7B-Instruct-AWQ")
tokenizer.save_pretrained("Qwen2.5-7B-Instruct-AWQ")
```

`quant_config` 中参数均对应前文概念：`w_bit=4` 为比特数，`q_group_size=128` 即第 3 节组粒度，`zero_point=True` 表示非对称量化。`quantize` 默认采用内置校准集统计各通道重要性（第 5 节 AWQ 核心步骤），亦可通过 `calib_data` 替换为自己场景语料。校准数据越接近真实流量，所保护重要通道越准确。

### 路径三 GGUF（llama.cpp）

制作分两步：先将 HF 权重转为未量化 F16 GGUF，再压缩至目标档位。保留中间产物，可继续压其他档位对比。

```bash
git clone https://github.com/ggml-org/llama.cpp
pip install -r llama.cpp/requirements.txt

# 第一步：HF 权重 -> F16 GGUF
python llama.cpp/convert_hf_to_gguf.py /path/to/Qwen2.5-7B-Instruct \
  --outfile qwen2.5-7b-f16.gguf

# 第二步：量化到第 8 节拆过的 Q4_K_M
./llama-quantize qwen2.5-7b-f16.gguf qwen2.5-7b-q4_k_m.gguf Q4_K_M
```

若需挂载第 8 节所述 imatrix，中间增加统计步骤，量化时附加：

```bash
# 用一份校准文本统计重要性
./llama-imatrix -m qwen2.5-7b-f16.gguf -f calibration.txt -o imatrix.dat
./llama-quantize qwen2.5-7b-f16.gguf qwen2.5-7b-q4_k_m.gguf Q4_K_M \
  --imatrix imatrix.dat
```

### 跑起来

产出后，启动仅需一行命令（完整流程见「模型部署与服务化」一书）：

```bash
# GPTQ / AWQ checkpoint 会被自动识别
vllm serve ./Qwen2.5-7B-Instruct-AWQ

# 在线压 FP8：BF16 checkpoint 加载时直接压，省掉离线步骤
vllm serve Qwen/Qwen2.5-7B-Instruct --quantization fp8

# llama.cpp 同样暴露 OpenAI-compatible API
llama-server -m qwen2.5-7b-q4_k_m.gguf --port 8000
```

第二条命令值得单独记录：`--quantization fp8` 允许将现成 BF16 模型于加载时在线压为 FP8，免去离线步骤。适合快速试验。正式上线仍建议使用离线校准过的 checkpoint：校准数据贴近真实流量，精度更有保障。

## 11. 量化的收益与代价

量化并非无代价的优化，我们将两侧账目并列审视。

收益何在？显存直接缩减（INT4 理论 4 倍）；Decode 属 memory-bound，每步搬运数据减少，吞吐相应提升；节省显存可换取更大模型、更长上下文或更高并发；新硬件上 FP8 / FP4 具备原生加速。

代价何在？质量下降。典型表现为困惑度微升，数学、代码等精度敏感任务掉分更显著。且**模型越小越敏感**——经验上，30B 以上模型压 4-bit 常近无损，7B 到 14B 用 GPTQ / AWQ 的 4-bit 通常可接受，3B 以下需谨慎，优先 8-bit 或 FP8。此外 per-group scale 增加元数据、部分 kernel 不支持混合精度需先反量化，均属轻微工程成本。

故上线前标准动作：依第 9 节表选格式、按第 10 节命令压一版，再据评测流程跑质量对比。「INT4 省下显存，是否值得微小分数损失」，仅能由你自身场景的 benchmark 回答。

## 小结

本章将「低比特」拆解为可独立回答的问题。压缩对象？权重（W）、activation（A）、KV Cache 为三个不同目标。压缩形态？INT4 / INT8 的均匀格，或 FP8 / FP4 的浮点格，后者契合「挤在 0 附近」的分布且新硬件原生支持。

粒度粗细？tensor / channel / group，outlier 决定不可过粗。误差如何补救？RTN 为底线；GPTQ 补误差、AWQ 保通道、SmoothQuant 移难度、QAT 直接训练。

格式选择？先定运行时再选格式：GPU 服务取 GPTQ / AWQ / FP8，端侧取 GGUF（Q4_K_M 一族），快速实验取 bitsandbytes。如何执行？llm-compressor 产 GPTQ / FP8，AutoAWQ 产 AWQ，llama.cpp 两步产 GGUF；vLLM 亦支持加载时在线压 FP8。

是否划算？小模型更敏感，上线前需用评测流程验证质量。

下一章延续 Decode 串行瓶颈展开：

> **每一步便宜了，但一次 forward 还是只能确认一个 Token。能不能一次确认多个？**

## 作业

练习时间到。三道题分别对应：粒度、非对称量化、解读模型名——皆为实际部署最先遭遇的三件事。

> **关于 AI 辅助**：可以让 AI 提示思路、拆解步骤，但不建议直接让 AI 完成题目。

### 作业 1：实现 per-group 量化并比较误差

作业 1 需实现 per-group 量化并对比误差。

操作方式为逐行将每 `group` 个相邻元素以组内 scale 量化再还原，随后与 per-tensor 比较误差。

**小提示**：组内 scale 是 `np.max(np.abs(block)) / 7`，之后 round、clip 到 `[-7, 7]` 再乘回。

In [ ]:
# 作业 1：per-group 量化 填空

np.random.seed(0)
W_test = np.random.randn(8, 32).astype(np.float32)
W_test[0] *= 10  # 制造一个 outlier 行

def qdq_group_cols(a, group=8):
    """逐行把每 group 个相邻元素用组内 scale 量化再还原"""
    out = np.zeros_like(a)
    for r in range(a.shape[0]):
        for c0 in range(0, a.shape[1], group):
            block = a[r, c0:c0 + group]
            # TODO：把下面三引号里的内容替换成你的代码
            """算组内 scale，round/clip 后乘回，写进 out[r, c0:c0+group]"""
    return out

err_tensor = np.mean(np.abs(W_test - qdq_tensor(W_test)))
err_group = np.mean(np.abs(W_test - qdq_group_cols(W_test)))
assert err_group < err_tensor, (err_group, err_tensor)
print("✅ 作业 1 通过：组内 outlier 只污染自己那组，误差更小")

### 作业 2：非对称量化（zero point）

作业 2 关于非对称量化（zero point）。

当数值几乎全处正半轴时（许多 Activation 即如此），对称量化会浪费半数格位。非对称量化将 `[min, max]` 整段映射至 `[0, 14]`，并记录 zero point 标示原点。

**小提示**：`scale = (max - min) / 14`，`zero_point = round(-min / scale)`；
量化 `q = round(x / scale) + zero_point` 后 clip 到 `[0, 14]`，反量化 `(q - zero_point) * scale`。

In [ ]:
# 作业 2：非对称量化 填空

x = np.array([2.0, 2.5, 3.0, 3.5, 4.0], dtype=np.float32)  # 全在正半轴
scale = (x.max() - x.min()) / 14
zero_point = int(np.round(-x.min() / scale))

def qdq_asym(a):
    """把 [min, max] 映射到 [0, 14] 的量化-反量化"""
    # TODO：把下面三引号里的内容替换成你的代码
    """q = round(a/scale) + zero_point 后 clip 到 [0,14]，再反量化回来"""

x_hat = qdq_asym(x)
assert np.max(np.abs(x - x_hat)) <= scale / 2 + 1e-6, (x, x_hat)
print("✅ 作业 2 通过：非对称量化把整段范围用满，误差不超过半个格子")

### 作业 3：读懂量化模型的名字

作业 3 要求解读量化模型名称。

Hugging Face 上的名称如 `Qwen2.5-7B-Instruct-GPTQ-Int4`，需能拆出方法与精度。

**小提示**：`Int8` 出现就是 8-bit，否则默认 4-bit；方法在 `GPTQ` / `AWQ` / `GGUF` 里找。

In [ ]:
# 作业 3：模型名拆解 填空

def parse_quant_name(name):
    """从量化模型名拆出 {'bits': int, 'method': str}"""
    # TODO：把下面三引号里的内容替换成你的代码
    """bits 取 Int8/Int4 中的数字（默认 4）；method 取 GPTQ/AWQ/GGUF 之一"""

info = parse_quant_name("Qwen2.5-7B-Instruct-GPTQ-Int4")
assert info == {"bits": 4, "method": "GPTQ"}, info
info2 = parse_quant_name("Llama-3-8B-AWQ")
assert info2 == {"bits": 4, "method": "AWQ"}, info2
print("✅ 作业 3 通过：看到量化模型名，你能马上说出方法和精度")